# 07 - Real Historical Retrieval with FAISS & Dense MiniLM Embeddings

## Overview
This notebook demonstrates dense vector retrieval over **2,245 historical resolved `@AppleSupport` conversations**.

### Architecture:
- **Embedding Model**: `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors)
- **Vector Store**: `faiss.IndexFlatIP` (Exact Cosine Similarity via L2-normalized Inner Product)
- **Data Source**: Strictly from `data/splits/train.jsonl` (Zero-Leakage Assurance: Test split and Golden Set excluded from the index)
- **Evaluation**: Intent Alignment Recall@1, Recall@3, Recall@5, and Mean Reciprocal Rank (MRR)

In [ ]:
import sys
import json
from pathlib import Path
import faiss
import numpy as np

base_dir = Path('..').resolve()
if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))

from app.services.retrieval.retriever import Retriever

summary_path = base_dir / 'models' / 'embedding_model' / 'index_summary.json'
with open(summary_path, 'r', encoding='utf-8') as f:
    summary = json.load(f)

print(f"Embedding Model:     {summary['model_name']}")
print(f"Embedding Dimension: {summary['embedding_dim']}")
print(f"Indexed Vectors:     {summary['total_cases_indexed']}")
print(f"Metric:              {summary['metric']}")
print(f"Intent Distribution: {json.dumps(summary['intent_distribution'], indent=2)}")

## 1. Interactive Nearest Neighbor Retrieval
Test real customer queries across multiple support domains and inspect the historical `@AppleSupport` resolution evidence.

In [ ]:
retriever = Retriever()

test_queries = [
    "My battery percentage drops from 80% to 20% in an hour after updating to iOS 11",
    "I forgot my Apple ID password and my account is disabled for security reasons",
    "My AirPods left earbud has very low sound and crackles during phone calls",
    "I was billed twice for Apple Music subscription this month"
]

for query in test_queries:
    print(f"\n{'='*75}")
    print(f"CUSTOMER INQUIRY: {query}")
    print('='*75)
    results = retriever.retrieve(query, top_k=2)
    for rank, case in enumerate(results, start=1):
        print(f"  Top {rank} [Score: {case.similarity:.4f}] [Intent: {case.metadata.get('intent')}]")
        print(f"    Historical Query: {case.customer_text[:100]}...")
        print(f"    Brand Resolution: {case.brand_response[:120]}...")

## 2. Empirical Benchmark Evaluation Metrics
Load empirical evaluation benchmarks computed by `scripts/evaluation/evaluate_retrieval.py` on the held-out test split and the golden set.

In [ ]:
benchmarks_path = base_dir / 'experiments' / 'retrieval_benchmarks.json'
with open(benchmarks_path, 'r', encoding='utf-8') as f:
    benchmarks = json.load(f)

import pandas as pd
rows = []
for key, bm in benchmarks['benchmarks'].items():
    rows.append({
        'Dataset': bm['dataset'],
        'Queries': bm['query_count'],
        'Recall@1': f"{bm['recall_at_1'] * 100:.2f}%",
        'Recall@3': f"{bm['recall_at_3'] * 100:.2f}%",
        'Recall@5': f"{bm['recall_at_5'] * 100:.2f}%",
        'MRR': f"{bm['mrr']:.4f}",
        'Mean Top-1 Sim': f"{bm['mean_top1_similarity']:.4f}"
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))